In [12]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import StructField, StructType, FloatType, IntegerType, StringType, DataType

import os
from dotenv import load_dotenv

#load environment variables
env = load_dotenv(dotenv_path='/mnt/d/linux/project/vars/.env')

#env vars for postgresql connection
mssqlsource = os.getenv('MSSQLSOURCE')
mssqluser = os.getenv('MSSQL_SPARK_USER')
mssqlpwd = os.getenv('MSSQL_SPARK_PWD')
mssqldriver = os.getenv('MSSQL_DRIVER')
sqlserverjar = os.getenv('SQLSERVER_JAR')
mssql_dwdest = os.getenv('MSSQL_CONN')

In [13]:
#creating sparksession
spark = SparkSession.\
            builder.\
            appName('teste_dimcolors').\
            config('spark.jars', f'{sqlserverjar}').\
            config('spark.driver.extraClassPath', f'{sqlserverjar}').\
            config('spark.executor.extraClassPath', f'{sqlserverjar}').\
        getOrCreate()
        
spark

In [14]:
dimcolors = ((spark.\
                read.\
                format('jdbc').\
                option('url', mssqlsource).\
                option('user', mssqluser).\
                option('password', mssqlpwd).\
                option('driver', mssqldriver).\
                option('query',
                       
                       """ 
                        select 

                            wrh_c.colorid,
                            wrh_c.colorname
                        
                        from warehouse.colors as wrh_c
                            left join
                            warehouse.colors_archive as wrh_ac
                        on wrh_c.colorid = wrh_ac.colorid

                       """))).load()

dimcolors.show()

+-------+-----------+
|colorid|  colorname|
+-------+-----------+
|      1|      Azure|
|      2|      Beige|
|      3|      Black|
|      4|       Blue|
|      5|   Charcoal|
|      6| Chartreuse|
|      7|       Cyan|
|      8| Dark Brown|
|      9| Dark Green|
|     10|    Fuchsia|
|     11|       Gold|
|     12| Steel Gray|
|     13|   Hot Pink|
|     14|     Indigo|
|     15|      Ivory|
|     16|      Khaki|
|     17|   Lavender|
|     18|Light Brown|
|     19|Light Green|
|     20|     Maroon|
+-------+-----------+
only showing top 20 rows



In [15]:

def selectcolors(spark = spark, connurl = mssqlsource , dbuser = mssqluser, dbpwd = mssqlpwd, dbdriver = mssqldriver):
    
    """ 
    
    this function selects data from colors table on warehouse schema
    All parameters are set by default variables that we create at the beginning.
    
    """
       
    dimension_colors = spark.\
                            read.\
                            format('jdbc').\
                            option('url', connurl).\
                            option('user', dbuser).\
                            option('password', dbpwd).\
                            option('driver', dbdriver).\
                            option('query',
                                   
                            """
                                select 

                                    wrh_c.colorid,
                                    wrh_c.colorname
                        
                                from warehouse.colors as wrh_c
                                
                                    left join
                                    
                                    warehouse.colors_archive as wrh_ac
                                    
                                on wrh_c.colorid = wrh_ac.colorid
                                   
                            """).\
                        load()
                            
    return dimension_colors

data = selectcolors()



In [16]:
def insertcolor(spark = spark, connurl = mssql_dwdest , dbuser = mssqluser, dbpwd = mssqlpwd, dbdriver = mssqldriver, data = data):
    
    data.\
        write.\
        format('jdbc').\
        option('url', connurl).\
        option('user', dbuser).\
        option('dbtable', 'dimension.colors').\
        option('mode', 'overwrite').\
        option('password', dbpwd).\
        option('driver', dbdriver).\
    save(mode='append')
    
insertcolor()